In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../src').resolve()))

In [ ]:
import download_data_mem
import processing
import utils
import pandas as pd
import numpy as np
import sys
import plot_radiosonde_ddu_map
import plot_radiosonde_linear 
import plot_radiosonde_skewt
import cluster_profile
import importlib
import matplotlib.pyplot as plt
from cluster_profile import ClusterProfiles
from datetime import date, datetime, timedelta
from pathlib import Path

In [ ]:
winter = download_data_mem.get_data(years=[2020,2021,2022,2023,2024,2025],months=[6,7,8],hours = [0]) #DJF / MAM / JJA / SON iwith 2 cons. year
#avant c'était juste 2023,2024,2025
#summer = download_data_mem.get_data(years=[2020],months=[12,1,2],hours = [0])
#transition = download_data_mem.get_data(years=[2020],months=[3,4,5,9,10,11],hours = [0])



In [ ]:
print("Winter:",len(winter))

In [ ]:
#Barplot:Count profiles per hour(0,12) date_per_sounding = winter[i]['header']['date'] (here only JJA)
date_per_sounding = pd.Series([winter[i]['header']['date'] for i in range(len(winter))])
date_per_sounding = date_per_sounding.to_frame()
date_per_sounding.rename(columns={0:'date'}, inplace=True)
date_per_sounding['date'] = date_per_sounding['date'].apply(lambda x: pd.to_datetime(x))
date_per_sounding['hour'] = date_per_sounding['date'].dt.hour
#hour = 12 if x closer to 12 than 24h warning if x=23 x-24= -1 closer to 0 than 12 but if x=1 x-0=1 closer to 0 than 12
#date_per_sounding['hour'] = date_per_sounding['hour'].apply(lambda x: 12 if abs(x-12) <= min(abs(x-24),abs(x-0)) else 0)
display(date_per_sounding['hour'].value_counts())
date_per_sounding['month'] = date_per_sounding['date'].dt.month
date_per_sounding['year'] = date_per_sounding['date'].dt.year
#barplot count profiles per hour
plt.figure(figsize=(5,3))
date_per_sounding['hour'].value_counts().sort_index().plot(kind='bar')
plt.xlabel('Hour of the day')
plt.ylabel('Count of profiles')
plt.title('Count of profiles per hour (JJA)')
plt.xticks(rotation=0)


In [ ]:
winter_int,winter_stats = utils.clean_and_interpolate(
    winter,
    z_grid=np.arange(50, 6000, 25),
    stitch=True,
    max_start_alt=50,   # profil doit démarrer sous 200 m
    min_coverage=0.85    # doit atteindre 85 % de 5000 m ≈ 4300 m
) #try to 5km-6km

print("Winter:",len(winter_int))

utils.plot_vertical_coverage(winter_int)

In [ ]:
#check wind spikes and temperature spikes
wind_spikes = []
temp_spikes = []
threshold =[]
dew_point =[]
pressure_nan = []
non_monotonic = []
discarded_b_of_pressure = []
discarded_b_of_len = []

for i in range(len(winter_stats)):
    if winter_stats[i]['wind_spikes'] != 0:
        wind_spikes.append(i)
    if winter_stats[i]['temp_spikes'] != 0:
        temp_spikes.append(i)
    if winter_stats[i]['threshold'] != 0:
        threshold.append(i)
    if winter_stats[i]['dewpoint'] != 0:
        dew_point.append(i)
    if winter_stats[i]['pressure_nan'] != 0:
        pressure_nan.append(i)
    if winter_stats[i]['non_monotonic_pressure'] != 0:
        non_monotonic.append(i)
    if winter_stats[i]['discarded_b_of_pressure'] != 0:
        discarded_b_of_pressure.append(i)
    if winter_stats[i]['discarded_b_of_len'] != 0:
        discarded_b_of_len.append(i)    
print("Wind spikes:",len(wind_spikes))
print("Temp spikes:",len(temp_spikes))
print("Threshold:",len(threshold))
print("Dew point:",len(dew_point))
print("Pressure NaN:",len(pressure_nan))
print("Non monotonic:",len(non_monotonic))
print("Discarded by pressure:",len(discarded_b_of_pressure))
print("Discarded by length:",len(discarded_b_of_len))

In [ ]:
importlib.reload(cluster_profile)
# INIT
cp = ClusterProfiles(variables=['t','td','t-td','u','v']) #variables only useful for build_X function, not for build_features
#voir d'autres variables : rh, rhi ,... ?

# BUILD MATRIX
X = cp.build_X(winter_int,T_anomaly=False) #T_anomaly=True, to use temperature anomaly instead of absolute temperature

# CLEANING
X_clean = cp.treat_nan(strategy="mean")

# NORMALISATION
X_norm = cp.normalize_shape(shape_vars=['t','td','t-td'],global_scale=True)
X_norm = cp.normalize_block(variables=['u','v'],method='z')



In [ ]:
size_per_var = int(1190/5)
#mean and std of each variable
for i,var in enumerate(['t','td','t-td','u','v']):
    var_data = X_norm[:,i*size_per_var:(i+1)*size_per_var]
    mean = np.mean(var_data)
    std = np.std(var_data)
    max_val = np.max(var_data)
    min_val = np.min(var_data)
    print(f"Variable: {var}, Mean: {mean:.2f}, Std: {std:.2f}, Max: {max_val:.2f}, Min: {min_val:.2f}")

In [ ]:
print("X_norm shape:", X_norm.shape)

In [ ]:
# PCA
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
pca = PCA()
pca.fit(X_norm)

# ============================================================
# BLOC 1 — Variance expliquée par composante
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(pca.explained_variance_ratio_) + 1),
       pca.explained_variance_ratio_,
       label='Variance par composante')
ax.plot(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_.cumsum(),
        color='red', marker='o', label='Variance cumulée')
ax.axhline(0.9, color='grey', linestyle='--', label='Seuil 90%')
ax.set_xlim(0,10)
ax.set_xlabel('Composante PCA')
ax.set_ylabel('Variance expliquée')
ax.set_title('Variance expliquée par composante PCA')
ax.legend()
plt.tight_layout()
plt.show()

# Affichez les valeurs numériques — regardez PC1
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {v:.3f}  (cumulé: {pca.explained_variance_ratio_[:i+1].sum():.3f})")



In [ ]:
#apply PCA
X_red= cp.apply_pca(n_components=4)

In [ ]:
def is_peak(silhouette, idx):
    """idx est la position dans la liste silhouette (0 = k=2, 1 = k=3, etc.)"""
    if idx <= 0 or idx >= len(silhouette) - 1:
        return False
    return silhouette[idx] > silhouette[idx - 1] and silhouette[idx] > silhouette[idx + 1]

from collections import Counter
import numpy as np

xth_best_k = 3       # rang dans le top à regarder
nb_seeds = 500
for xth_best_k in [1,2,3]:  # on peut tester plusieurs rangs dans le top
    best_k_list = []       # meilleur k absolu par seed
    peak_k_list = []       # tous les k qui sont des pics locaux, par seed
    for seed in range(nb_seeds):
        silhouette = cp.find_nb_of_clusters(X_red, max_k=15, seed=seed, plot=False)
        # silhouette[i] correspond à k = i+2
        
        # Meilleur k absolu (rang xth_best_k)
        sorted_indices = np.argsort(silhouette)  # indices triés par silhouette croissant
        best_idx = sorted_indices[-xth_best_k]   # indice dans la liste silhouette
        best_k = best_idx + 2                    # k correspondant
        best_k_list.append(best_k)
        
        # Pics locaux : tous les k qui sont un maximum local dans la courbe
        peaks_this_seed = [i + 2 for i in range(len(silhouette)) if is_peak(silhouette, i)]
        peak_k_list.append(peaks_this_seed)

    # --- Analyse des meilleurs k ---
    k_counts = Counter(best_k_list)
    print("=== Top k par silhouette (rang", xth_best_k, ") ===")
    for k, count in k_counts.most_common(5):
        # Compter combien de fois ce k était aussi un pic local dans la même seed
        n_peak = sum(k in peaks for peaks in peak_k_list)
        print(f"  k={k}: apparaît {count}/{nb_seeds} fois, est pic local dans {n_peak}/{nb_seeds} seeds")

    # --- Analyse des pics locaux ---
    all_peaks = [k for peaks in peak_k_list for k in peaks]
    peak_counts = Counter(all_peaks)
    print("\n=== Fréquence des pics locaux (toutes seeds) ===")
    for k, count in peak_counts.most_common(10):
        print(f"  k={k}: pic local dans {count}/{nb_seeds} seeds")

    # --- Résumé ---
    print("\n=== Recommandation ===")
    # k qui est à la fois fréquemment choisi ET fréquemment un pic local
    for k in range(2, 16):
        n_best = k_counts.get(k, 0)
        n_peak = sum(k in peaks for peaks in peak_k_list)
        if n_peak >= 20 or n_best >= 10:
            print(f"  k={k}: choisi {n_best}/{nb_seeds} fois | pic local {n_peak}/{nb_seeds} seeds")

In [ ]:
from sklearn.metrics import silhouette_score
best_score = -1
best_labels = None
best_seed = None

for seed in range(200):
    km = KMeans(n_clusters=6, n_init='auto', random_state=seed)
    labels = km.fit_predict(X_red)
    s = silhouette_score(X_red, labels)
    if s > best_score:
        best_score = s
        best_labels = labels
        best_seed = seed

print(f"Best seed: {best_seed}, silhouette: {best_score:.4f}")

In [ ]:
# CHOIX DU NOMBRE DE CLUSTERS
cp.find_nb_of_clusters(X_red,max_k=15,seed=24)

In [ ]:
import numpy as np
n = len(X_red)
co_matrix = np.zeros((n, n))
n_runs = 200

for seed in range(n_runs):
    km = KMeans(n_clusters=6, n_init=20, random_state=seed)
    labels = km.fit_predict(X_red)
    for k in range(6):
        idx = np.where(labels == k)[0]
        for i in idx:
            co_matrix[i, idx] += 1

co_matrix /= n_runs
# co_matrix[i,j] = fraction des runs où i et j sont dans le même cluster
# Valeurs proches de 1 = paire stable, proches de 0.5 = frontière ambiguë

In [ ]:
#count how many pairs have co_matrix[i,j] > 0.8 and how many have co_matrix[i,j] < 0.2 and how many have co_matrix[i,j] between 0.4 and 0.6
stable_pairs = (np.sum(co_matrix > 0.8)-579)/2
separated_pairs = np.sum(co_matrix < 0.2)/2
boundary_pairs = np.sum((co_matrix >= 0.3) & (co_matrix <= 0.7))/2
print(f"Co-assigned pairs (>0.8): {stable_pairs}, Separated pairs (<0.2): {separated_pairs}, Ambiguous pairs (0.3-0.7): {boundary_pairs}")

In [ ]:
nb_pair_unique = 167331

#percentage of stable pairs, separated pairs and ambiguous pairs
print(f"Stable pairs: {stable_pairs/nb_pair_unique:.2%}, Separated pairs: {separated_pairs/nb_pair_unique:.2%}, Ambiguous pairs: {boundary_pairs/nb_pair_unique:.2%}")


In [ ]:
# FIT FINAL
labels = cp.fit_kmeans(k=6,seed=24,n_init='auto')
#plt.hist(labels, bins=np.arange(-0.5, 4.5, 1), rwidth=0.8)
plt.figure(figsize=(8,6))

for i in range(labels.min(), labels.max()+1):
    plt.scatter(
        X_red[labels == i, 0],
        X_red[labels == i, 1],
        label=f'Cluster {i}'
    )

plt.legend(title='Clusters')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('K-means Clusters in PCA Space')
plt.show()
#silhouette
kmeans_silhouette = silhouette_score(X_red, labels)
print(f"KMeans Silhouette Score: {kmeans_silhouette:.4f}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
# BLOC 2 — Projection PC1 vs PC2 colorée par cluster k=3-4

# Labels du clustering d'avant k=3/4/.. 
labels = cp.labels  # adaptez selon votre attribut

# Re-fit PCA à 2 composantes pour la visualisation
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_norm)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=labels, cmap='Set1', alpha=0.6, s=40)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title(f'Projection PCA 2D — clusters k={len(np.unique(labels))}')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
z_grid = np.arange(50,6000,25)
fig, df = cp.plot_pca_loadings(z_grid, n_components=4)

In [ ]:
print(df['PC1'].abs().sort_values(ascending=False).head(10))
print(df['PC2'].abs().sort_values(ascending=False).head(10))
print(df['PC3'].abs().sort_values(ascending=False).head(10))
print(df['PC4'].abs().sort_values(ascending=False).head(10))

In [ ]:

print(df['PC1'].abs().sort_values(ascending=False).head(10)
,'\n',df['PC1'].abs().sort_values(ascending=True).head(10))

print('\n')

print(df['PC2'].abs().sort_values(ascending=False).head(10)
,'\n',df['PC2'].abs().sort_values(ascending=True).head(10))

print('\n')

print(df['PC3'].abs().sort_values(ascending=False).head(10)
,'\n',df['PC3'].abs().sort_values(ascending=True).head(10))

In [ ]:
cp.plot_x_first_m_profiles(winter_int,200)

In [ ]:
z_grid = np.arange(50,6000,25)
cp.plot_cluster_minipages(winter_int, z_grid,quantiles=(0.25,0.75),ylim=(0, 6000))

In [ ]:
mean_df,std_df, all_mean,all_std, df = cp.cluster_summary_full(winter_int) #all_mean et all_std sont les moyennes et std sur l'ensemble des profils, sans groupby cluster

In [ ]:
from scipy import stats

vars_of_interest = [
    # Vent par couche
    "ws_sfc", "ws_500", "ws_1km", "ws_3km",
    "ws_mean_0_1km", "ws_mean_1_3km",
    "u_sfc", "v_sfc",
    "u_1km", "v_1km",
    "u_3km", "v_3km",
    "wind_turning_0_3km",
    "shear_0_500",
    # Jet
    "jet_nose_speed", "jet_nose_height",
    "jet_shear_below", "jet_shear_above",
    # Thermique
    "T_sfc_C",
    "T_mean_0_500_C", "T_mean_0_1km_C", "T_mean_1_3km_C",
    "theta_sfc",
    "T_range_0_1km",
    "N2_surface",
    # Inversion
    "inversion_strength", "inversion_height",
    "max_grad_0_500",
    # Lapse rates larges
    "lapse_0_500", "lapse_0_2km", "lapse_1_3km", "lapse_3_6km",
    # Profil lapse rate 0-2000m
    "lapse_z0_200", "lapse_z200_400", "lapse_z400_600",
    "lapse_z600_800", "lapse_z800_1000",
    "lapse_z1000_1200", "lapse_z1200_1400",
    "lapse_z1400_1600", "lapse_z1600_1800", "lapse_z1800_2000",
    # Humidité
    "dd_0_500", "dd_1_3km",
    "RH_mean_0_1km", "RH_mean_1_3km",
    "RH_max", "z_RH_max",
    "IWV",
]

print(f"{'Variable':<25} {'F':>8} {'p':>10} {'Significant':>12}")
print("-"*58)
for var in vars_of_interest:
    groups = [df[df["cluster"] == k][var].dropna() for k in range(6)]
    F, p = stats.f_oneway(*groups)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{var:<25} {F:>8.1f} {p:>10.4f} {sig:>12}")

In [ ]:
# Effect size : (mean_cluster - mean_global) / std_global
global_mean = df.drop(columns="cluster").mean()
global_std  = df.drop(columns="cluster").std()

effect_size = (mean_df - global_mean) / global_std

print(effect_size[vars_of_interest].round(2).to_string())

In [ ]:
# Ratio std_cluster / std_total
relative_std = std_df.div(all_std, axis=1)

# Affichage lisible
print("\nRelative std (cluster_std / total_std) — < 1 means cluster is tighter than dataset")
print("="*80)
# Garde seulement les variables intéressantes
vars_of_interest = [
    "ws_sfc", "ws_3km", "u_sfc", "v_sfc", "u_3km", "v_3km",
    "wind_turning_0_3km",
    "IWV", "RH_mean_0_1km", "RH_mean_1_3km",
    "N2_surface",
    "lapse_z0_200", "lapse_z400_600", "lapse_z600_800", "lapse_z800_1000",
    "jet_nose_speed", "jet_nose_height",
    "T_sfc_C", "inversion_strength"
]

print(relative_std[vars_of_interest].round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Extraire les colonnes lapse profile
lapse_cols = [c for c in mean_df.columns if c.startswith('lapse_z')]
lapse_df = mean_df[lapse_cols]

# Noms de lignes : labels physiques des clusters
cluster_labels = {
    0: 'C0 — ',
    1: 'C1 — ',
    2: 'C2 — ',
    3: 'C3 — ',
    4: 'C4 — ',
    5: 'C5 — ',
    6: 'C6 — ',
    7: 'C7 — ',
}
row_labels = [cluster_labels[i] for i in lapse_df.index]

# Noms de colonnes lisibles (0-200, 200-400, ...)
col_labels = [c.replace('lapse_z', '').replace('_', '–') + 'm' for c in lapse_cols]

# Valeurs en K/100m pour lisibilité
values = lapse_df.values * 100  # K/m → K/100m

# Colormap divergente centrée sur 0
vmax = np.abs(values).max()
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.cm.RdBu_r  # rouge = inversion, bleu = lapse normal

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(values, cmap=cmap, norm=norm, aspect='auto')

# Axes
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=10, rotation=30, ha='right')
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=10)

# Annotations dans les cellules
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        val = values[i, j]
        color = 'white' if abs(val) > vmax * 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8.5, color=color)

# Colorbar
cbar = fig.colorbar(im, ax=ax, orientation='vertical', pad=0.02, shrink=0.85)
cbar.set_label('Lapse rate (K/100m)', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Ligne de séparation à 0 sur la colorbar
cbar.ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

ax.set_title('Lapse rate profile 0–2000 m by cluster\n'
             'Red = inversion (dT/dz > 0)   |   Blue = normal lapse (dT/dz < 0)',
             fontsize=11, pad=12)

plt.tight_layout()
#plt.savefig('lapse_heatmap.pdf', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig=cp.plot_cluster_carpet(winter_int, season_months=[5, 6, 7, 8,9], years=[2020,2021,2022,2023,2024,2025],sounding_width=0.5)

In [ ]:
fig, stats = cp.plot_interannual_consistency(
    winter_int,
    season_months=[5, 6, 7, 8, 9],
    window=7,           # fenêtre 7 jours — ~20 bins pour la saison
    min_soundings=3,    # bin ignoré si < 3 sondages (gaps, début/fin de saison)
)

In [ ]:
stats

In [ ]:
cp_uv = ClusterProfiles(variables=['u','v'])
X_uv = cp_uv.build_X(winter_int,T_anomaly=False)
X_clean_uv = cp_uv.treat_nan(strategy="mean")
X_norm_uv = cp_uv.normalize_block(['u','v'],method='z')
X_red_uv = cp_uv.apply_pca(n_components=4)
#cp_uv.find_nb_of_clusters(X_red_uv,max_k=15)
labels_uv = cp_uv.fit_kmeans(k=6,seed=24,n_init='auto')
from sklearn.metrics import adjusted_rand_score
score = adjusted_rand_score(labels, labels_uv)
print(f"ARI = {score:.3f}")  # 1.0 = partitions identiques


In [ ]:
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix
import numpy as np

# Aligner les clusters avec l'algorithme hongrois
cm = confusion_matrix(labels, labels_uv)
row_ind, col_ind = linear_sum_assignment(-cm)
accuracy = cm[row_ind, col_ind].sum() / len(labels)
print(f"Alignment accuracy: {accuracy:.3f}")

In [ ]:
cp_uv.cluster_summary_full(winter_int)

In [ ]:
#2 plots: PCA with labels and labels_uv
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
# PCA avec labels k-means sur t, td, t-td, u, v
scatter1 = ax[0].scatter(X_red[:, 0], X_red[:, 1], c=labels, cmap='Set1', alpha=0.6, s=40)
ax[0].set_title('All features')
ax[0].set_xlabel('PC1')
ax[0].set_ylabel('PC2')
ax[0].grid(alpha=0.3)
# PCA avec labels k-means sur t, td, t-td seulement
scatter2 = ax[1].scatter(X_red_uv[:, 0], X_red_uv[:, 1], c=labels_uv, cmap='Set1', alpha=0.6, s=40)
ax[1].set_title('U and V only')
ax[1].set_xlabel('PC1')
ax[1].set_ylabel('PC2')
ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig = cp.plot_date_distribution(winter_int)

In [ ]:
# Exemple été austral
fig, summary = cp.plot_cluster_seasonal_evolution(
    winter_int, season_months=[6, 7, 8])

In [ ]:
fig, df_agg = cp.plot_feature_seasonal_evolution(
    winter_int,
    season_months=[6, 7, 8],  # hiver austral
    features = 'all',
    n_cols = 3
)

In [ ]:
df_agg.head(25)

In [ ]:
cp.plot_dendrogram(show_cut_at_k=4)
cp.silhouette_hierarchical(max_k=15,linkage_method = 'single')

In [ ]:
labels = cp.fit_hierarchical(k=4)
#plot pca with labels
plt.scatter(X_red[:, 0], X_red[:, 1], c=labels, cmap='Set1', alpha=0.6, s=40)
plt.title('Hierarchical Clustering (single linkage)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.3)
plt.show()
cp.plot_cluster_minipages(winter_int, z_grid,quantiles=(0.25,0.75),ylim=(0, 6000))
cp.cluster_summary_full(winter_int)

In [ ]:
fig = cp.plot_date_distribution(winter_int)

In [ ]:
fig,summary = cp.plot_cluster_seasonal_evolution(
    winter_int, season_months=[6, 7, 8])

In [ ]:
fig, df_agg = cp.plot_feature_seasonal_evolution(
    winter_int,
    season_months=[6, 7, 8],  # hiver austral
    features = 'all',
    n_cols = 3
)

In [ ]:
df_agg

In [ ]:
labels_wind, labels_therm, labels = cp.fit_twostage(
    k_wind=6,
    n_pca_wind=4,
    max_k_therm=5,
    cumvar_therm=0.90,
    min_silhouette=0.20,
    min_cluster_size=10,   # ← chaque sous-cluster doit avoir au moins 10 profils
    n_init='auto',
    seed=81
)
fig=cp.plot_twostage_scatter()           # espace PCA vent (régimes + sous-types)
fig2=cp.plot_twostage_thermal_scatter()   # espace PCA thermique (un panel par régime vent)

In [ ]:
z_grid = np.arange(50,6000,25)
cp.plot_cluster_minipages(winter_int, z_grid,quantiles=(0.25,0.75),ylim=(0, 6000))

In [ ]:
cp.cluster_summary_full(winter_int)

In [ ]:
fig=cp.plot_cluster_carpet(winter_int, season_months=[5, 6, 7, 8,9], years=[2020,2021,2022,2023,2024,2025],sounding_width=0.5)
fig, stats = cp.plot_interannual_consistency(
    winter_int,
    season_months=[5, 6, 7, 8, 9],
    window=7,           # fenêtre 7 jours — ~20 bins pour la saison
    min_soundings=3,    # bin ignoré si < 3 sondages (gaps, début/fin de saison)
)